In [ ]:
!pip install pyDOE

In [ ]:
import tensorflow as tf
import numpy as np
from pyDOE import lhs
from google.colab import drive, files
import matplotlib.pyplot as plt

from itertools import product

In [ ]:
plt.rcParams.update({'font.size': 14})

### Define the ICs

In [ ]:
# Boundaries of the computational domain
xleft, xright = -np.pi, np.pi
t0, tfinal = 0, 2.0

In [ ]:
# KdV coefficients
eps = 1.2e-1
mu = 8e-4

In [ ]:
# How many sensor points to use
N_sensor_points = 100

In [ ]:
def u0(t, x, amplitude, phase):
  return amplitude/2/tf.math.cosh(tf.sqrt(amplitude)/2*(5*x - amplitude*t/10-phase))**2

In [ ]:
def defineCollocationPoints(N_pde=10000, N_u=100):

  # Uniform random sampling for PDE points
  tx_min = np.array([t0, xleft])
  tx_max = np.array([tfinal, xright])
  pde_points = tx_min + (tx_max - tx_min)*lhs(2, N_pde)
  t_pde = pde_points[:, 0]
  x_pde = pde_points[:, 1]

  # Initial data sample
  x_points = np.linspace(xleft, xright, N_sensor_points)
  t_points = 0*x_points

  usensors = []
  pdes = np.column_stack([t_pde, x_pde]).astype(np.float32)

  for i in range(N_u):

    amplitudes = 4*np.random.rand() + 1
    phases = -8*np.random.rand() + 4

    # Generate the data for the branch net
    u0curr = u0(np.expand_dims(t_points, axis=1), np.expand_dims(x_points, axis=1), amplitudes, phases)

    # Generate the data for the trunk net
    amplitudes = np.expand_dims(amplitudes, axis=0)
    phases = np.expand_dims(phases, axis=0)
    usensors.append(np.row_stack([u0curr, amplitudes, phases]))

  usensors = np.transpose(np.column_stack(usensors).astype(np.float32), (1,0))

  return (pdes, usensors)

In [ ]:
N_pde, N_u = 10000, 50
(pdes, usensors) = defineCollocationPoints(N_pde, N_u)

### DeepONet model

In [ ]:
class PeriodicBCs(tf.keras.layers.Layer):
  def __init__(self, name=None, **kwargs):
    super(PeriodicBCs, self).__init__(name=name, **kwargs)

  def call(self, inputs):
    return tf.concat((tf.cos(inputs), tf.sin(inputs)), axis=1)


# Define the normalization layer
class Normalize(tf.keras.layers.Layer):
  def __init__(self, xmin, xmax, name=None, **kwargs):
    super(Normalize, self).__init__(name=name)
    self.xmin = xmin
    self.xmax = xmax
    super(Normalize, self).__init__(**kwargs)

  def call(self, inputs):
    return 2.0*(inputs-self.xmin)/(self.xmax-self.xmin)-1.0

  def get_config(self):
    config = super(Normalize, self).get_config()
    config.update({'xmin': self.xmin, 'xmax': self.xmax})
    return config

In [ ]:
def mlp_network(inp, n_layers, n_units):

  h = inp
  for i in range(n_layers-1):
    h = tf.keras.layers.Dense(n_units, activation='tanh')(h)
  out = tf.keras.layers.Dense(n_units, activation='linear')(h)
  return out

In [ ]:
# Architecture
n_units = 40
n_layers = 6

# Branch net
inp_u = tf.keras.layers.Input(shape=(N_sensor_points, ))
branch_net = mlp_network(inp_u, n_layers, n_units)

# Pre net
pre_net = mlp_network(inp_u, n_layers, n_units)
scale = tf.keras.layers.Dense(3, use_bias=False)(pre_net)
shift = tf.keras.layers.Dense(3)(pre_net)

# Trunk net
inp_t = tf.keras.layers.Input(shape=(1,))
b_t = Normalize(t0, tfinal)(inp_t)

inp_x = tf.keras.layers.Input(shape=(1,))
periodic_x = PeriodicBCs()(inp_x)

inp_tx = tf.keras.layers.Concatenate()([b_t, periodic_x])

# Apply pre-net to prepared trunk net input
scale_inp_tx = tf.keras.layers.Multiply()([scale, inp_tx])
shift_scale_inp_tx = tf.keras.layers.Add()([shift, scale_inp_tx])

# Actual trunk net
# trunk_net = mlp_network(shift_scale_inp_tx, n_layers, n_units)
trunk_net = mlp_network(inp_tx, n_layers, n_units)

out = tf.keras.layers.Multiply()([branch_net, trunk_net])
out = tf.keras.layers.Dense(1)(out)

model = tf.keras.models.Model([inp_t, inp_x, inp_u], out)
model.summary()

In [ ]:
tf.keras.utils.plot_model(model)

In [ ]:
@tf.function(reduce_retracing=True)
def train_network(pdes, sensorData, opt):

  # PDE points
  t, x = pdes[:,:1], pdes[:,1:2]
  u_in, amplitude, phase = sensorData[:,:N_sensor_points], sensorData[:,-2:-1], sensorData[:,-1:]

  # Outer gradient for tuning network parameters
  with tf.GradientTape() as tape:

    with tf.GradientTape() as tape1:
      tape1.watch(t), tape1.watch(x)
      with tf.GradientTape() as tape2:
        tape2.watch(t), tape2.watch(x)
        with tf.GradientTape() as tape3:
          tape3.watch(t), tape3.watch(x)
          u = u0(0*t, x, amplitude, phase)*(1-t/tfinal) + t/tfinal*model([t, x, u_in])
        [ut, ux] = tape3.gradient(u, [t, x])
      uxx = tape2.gradient(ux, x)
    uxxx = tape1.gradient(uxx, x)

    # KdV equation
    eqn = ut + eps*u*ux + mu*uxxx
    eqn = tf.where(tf.math.is_nan(eqn), 0.0, eqn)

    # Loss
    loss = tf.reduce_mean(tf.square(eqn))*100.0

  # Model gradients and update
  grads = tape.gradient(loss, model.trainable_variables)

  opt.apply_gradients(zip(grads, model.trainable_variables))

  return loss

In [ ]:
# Helper function to map all trunk-net collocation points to all branch-net ICs
def mapBatches(x, y):
  zipped = list(product(list(map(tuple, x)), list(map(tuple, y))))
  zipped = np.array(zipped, dtype=object)
  x = tf.convert_to_tensor(np.stack(zipped[:,0]), dtype=tf.float32)
  y = tf.convert_to_tensor(np.stack(zipped[:,1]), dtype=tf.float32)
  return x, y

In [ ]:
def trainDeepOnet(pde_points, u_sensors, epochs=10000):

  # Number of ICs and total number of PDE collocation points
  N_u = pde_points.shape[0]
  N_pde_points = pde_points.shape[1]

  # Batch size for PDE points
  batch_size_pde = 1000

  # Batch size for initial conditions
  batch_size_u = 50

  ds_u = tf.data.Dataset.from_tensor_slices(u_sensors)
  ds_u = ds_u.cache().shuffle(N_u).batch(batch_size_u, drop_remainder=True)

  ds_pde = tf.data.Dataset.from_tensor_slices(pde_points)
  ds_pde = ds_pde.cache().shuffle(N_pde_points).batch(batch_size_pde, drop_remainder=True)

  # Optimizer and learning rate to be used
  lr = 1e-3
  opt = tf.keras.optimizers.Adam(lr)

  # lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
  #       1.2e-2, decay_steps = 70, decay_rate = 0.95, staircase=False)
  # opt = tf.keras.optimizers.Adam(lr_schedule, beta_1 = 0.95, beta_2 = 0.99)

  epoch_loss = np.zeros(epochs)

  # Main training loop
  for i in range(epochs):

    nr_batches = 0

    # Training for that epoch
    for sensorData in ds_u:

      for pdes in ds_pde:

        # Cartesian product of input batches
        sensorData_batched, pdes_batched = mapBatches(sensorData.numpy(), pdes.numpy())

        # Train the network
        loss = train_network(pdes_batched, sensorData_batched, opt)
        # loss = train_network(pdes, sensorData, opt)

        epoch_loss[i] += loss
        nr_batches += 1

    # Get total epoch loss
    epoch_loss[i] /= nr_batches

    if (np.mod(i, 10)==0):
      print(f"Total loss {i}th epoch: {epoch_loss[i]: 6.4f}.")

  return epoch_loss

In [ ]:
epochs = 1000
loss = trainDeepOnet(pdes, usensors, epochs)

In [ ]:
plt.semilogy(range(0, epochs), loss)
plt.xlabel('epochs')
plt.ylabel('loss')
plt.savefig('loss.png')
plt.grid()

In [ ]:
# Grid where to evaluate the model
l, n = 50, N_sensor_points
t = np.linspace(t0, tfinal, l)
x = np.linspace(xleft, xright, n)
T, X  = np.meshgrid(t, x, indexing='ij')

In [ ]:
# Sample a new IC
amplitudes = 3.0*np.random.rand() + 1.0
phases = -np.pi*np.random.rand() + np.pi/2

amplitudes = 2.0
phases = 0.0

x_points = np.linspace(xleft, xright, N_sensor_points)
t_points = 0*x_points
u0curr = u0(np.expand_dims(t_points, axis=1), np.expand_dims(x_points, axis=1), amplitudes, phases)

# Generate the data for the trunk net
amplitudes = np.expand_dims(amplitudes, axis=0)
phases = np.expand_dims(phases, axis=0)

sensorData = np.row_stack([u0curr, amplitudes, phases])
sensorData = np.transpose(sensorData, (1,0))

usensor, amplitude, phase = sensorData[:,:N_sensor_points], sensorData[:,-2], sensorData[:,-1]

u = model([np.expand_dims(T.flatten(), axis=1),
           np.expand_dims(X.flatten(), axis=1),
           usensor])[:,0]
u = u0(0*T.flatten(), X.flatten(), amplitude, phase)*(1-T.flatten()/tfinal) + T.flatten()/tfinal*u.numpy()

u = np.reshape(u, (l, n))

In [ ]:
uTrue = u0(T, X, amplitude, phase)

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(131)

plt.contourf(T, X, u, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('NN solution')
plt.colorbar()


plt.subplot(132)
plt.contourf(T, X, uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('True solution')
plt.colorbar()

plt.subplot(133)
plt.contourf(T, X, u-uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title(f'MAE: {tf.reduce_mean(tf.abs(u-uTrue)): 2.4f}')
plt.colorbar()

In [ ]:
plt.plot(x, u[-1,],'k')
plt.plot(x, uTrue[-1,],'r--')
plt.grid()
plt.legend(['NN','GT'])

### Time stepping with DeepOnet

In [ ]:
steps = 10

In [ ]:
uinit = u0(T.flatten(), X.flatten(), amplitude, phase)

u = []
for i in range(steps):
  u_c = model([np.expand_dims(T.flatten(), axis=1),
               np.expand_dims(X.flatten(), axis=1),
               usensor])[:,0]
  u_c = np.reshape(u_c, (l, n))

  # Reconstruct solution
  uinit = np.repeat(usensor, repeats=l, axis=0)

  # For form u(t,x) = u0(x) + t/tf*v(t,x)
  # u_c = uinit + T/tfinal*u_c

  # # For form u(t,x) = u0(x)*(1-t/tf) + t/tf*v(t,x)
  u_c = uinit*(1-T/tfinal) + T/tfinal*u_c

  u.append(u_c)

  # Next initial condition
  usensor = u_c[-1:,:]

u = np.concatenate(u)

In [ ]:
# Exact solution
l, n = 50, N_sensor_points
t = np.linspace(t0, steps*tfinal, steps*l)
x = np.linspace(xleft, xright, n)
TAll, XAll  = np.meshgrid(t, x, indexing='ij')

uTrue = u0(TAll, XAll, amplitude, phase)

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(131)

plt.contourf(TAll, XAll, u, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('NN solution')
plt.colorbar()


plt.subplot(132)
plt.contourf(TAll, XAll, uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('True solution')
plt.colorbar()

plt.subplot(133)
plt.contourf(TAll, XAll, u-uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title(f'MAE: {tf.reduce_mean(tf.abs(u-uTrue)): 2.4f}')
plt.colorbar()

In [ ]:
plt.plot(x, u[-1,],'k')
plt.plot(x, uTrue[-1,],'k--')
plt.grid()
plt.legend(['NN','GT'])